In [ ]:
# import packages
import pandas as pd
import glob

In [ ]:
# path to all crime data
file_paths = glob.glob("../data/*/*.csv") # path to all crime data

# function that executes preprocessing
def preprocess(df):

    """Repeat the preprocess for each police force crime data"""

    # standardise column names
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    # check for null values
    df = df.drop(columns=["context"], inplace=False) # remove context column as it is empty
    df = df.dropna(subset=["longitude"], inplace=False) # remove any rows with missing location, as it is curcial for analysis
    
    # convert month column into datetime data type and split it into each column, month and year
    df["month"] = pd.to_datetime(df["month"]) # convert to datetime
    df["year"] = df["month"].dt.year # seperate year into new column called year
    df["month_num"] = df["month"].dt.month # seperate month number into new column called month_num

    # remove duplicates
    with_id = df[df["crime_id"].notnull()] # create a dataframe with rows that have a crime_id to check for duplicates
    without_id = df[df["crime_id"].isnull()] # create a dataframe with rows that is missing crime_id to check for duplicates

    with_id = with_id.sort_values(
    by = "last_outcome_category",
    key = lambda x: x == "Status update unavailable" # push "Status update unavailable to the bottom" prioritising data with an actual outcome category
    )
    with_id = with_id.drop_duplicates(subset=["crime_id"], keep="first", inplace=False) # remove duplicates from rows with crime_id
    without_id = without_id.drop_duplicates(subset = ["lsoa_code", "crime_type", "month"], keep="first", inplace=False) # remove assumed duplicates from missing crime_id rows

    df = pd.concat([with_id, without_id], ignore_index = True) # join the split with and without dataset again after removing duplicates

    return df

cleaned_dfs = [] # empty df for cleaned combined crime data

for file_path in file_paths: # loop through every csv files
    df = pd.read_csv(file_path) # read the current csv file
    cleaned_df = preprocess(df) # apply the preprocess function
    cleaned_dfs.append(cleaned_df) # store the cleaned df inside a list for later merging

combined_df = pd.concat(cleaned_dfs, ignore_index = True) # combine all cleaned df into one large df

combined_df.to_csv("../data/clean_crime_data.csv", index = False) # export the combined_df as a csv file


# combined crime data information
print("Final dataset shape:", combined_df.shape)

print("\nColumn info:")
print(combined_df.info())

print("\nSample rows:")
print(combined_df.head())

print("\nMissing values(sum):")
print(combined_df.isnull().sum())

print("\nMissing values(percentage):")
print(round(combined_df.isnull().mean(),2))

print("\nCrime types:")
print(combined_df["crime_type"].value_counts())

Final dataset shape: (3782624, 13)

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3782624 entries, 0 to 3782623
Data columns (total 13 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   crime_id               object        
 1   month                  datetime64[ns]
 2   reported_by            object        
 3   falls_within           object        
 4   longitude              float64       
 5   latitude               float64       
 6   location               object        
 7   lsoa_code              object        
 8   lsoa_name              object        
 9   crime_type             object        
 10  last_outcome_category  object        
 11  year                   int32         
 12  month_num              int32         
dtypes: datetime64[ns](1), float64(2), int32(2), object(8)
memory usage: 346.3+ MB
None

Sample rows:
                                            crime_id      month  \
0  e7b720d0e1302d2d06db7

# **Population csv file**

In [48]:
# import population csv file
pop = pd.read_csv("../data/pop21.csv", skiprows=3)

# standardise column names
pop.columns = pop.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)

# columns to convert
cols_to_convert = [
    "total",
    "f0_to_15",
    "f16_to_29",
    "f30_to_44",
    "f45_to_64",
    "f65_and_over",
    "m0_to_15",
    "m16_to_29",
    "m30_to_44",
    "m45_to_64",
    "m65_and_over"
]

# clean and convert
for col in cols_to_convert:
    pop[col] = (
        pop[col]
        .astype(str)
        .str.strip()
        .str.replace(",", "", regex=False)   # remove commas
        .replace({"": pd.NA, "nan": pd.NA, "-": pd.NA})
    )
    pop[col] = pd.to_numeric(pop[col], errors="coerce").astype("Int64")

pop.to_csv("../data/clean_pop.csv", index = False) # export the combined_df as a csv file

# combined crime data information
print("Final dataset shape:", pop.shape)

print("\nColumn info:")
print(pop.info())

print("\nSample rows:")
print(pop.head())

print("\nMissing values(sum):")
print(pop.isnull().sum())

Final dataset shape: (35672, 15)

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   lad_2021_code   35672 non-null  object
 1   lad_2021_name   35672 non-null  object
 2   lsoa_2021_code  35672 non-null  object
 3   lsoa_2021_name  35672 non-null  object
 4   total           35672 non-null  Int64 
 5   f0_to_15        35672 non-null  Int64 
 6   f16_to_29       35672 non-null  Int64 
 7   f30_to_44       35672 non-null  Int64 
 8   f45_to_64       35672 non-null  Int64 
 9   f65_and_over    35672 non-null  Int64 
 10  m0_to_15        35672 non-null  Int64 
 11  m16_to_29       35672 non-null  Int64 
 12  m30_to_44       35672 non-null  Int64 
 13  m45_to_64       35672 non-null  Int64 
 14  m65_and_over    35672 non-null  Int64 
dtypes: Int64(11), object(4)
memory usage: 4.5+ MB
None

Sample rows:
  lad_2021_code lad_2021_n